# 23 — Umpire accuracy by season

How often did the plate umpire's original ball/strike call agree with the strike zone, 2021–2026?

Note: 2021–2025 use the front-of-plate location and the operator-set zone.
2026 uses the Savant midpoint location and the ABS zone.

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table

TABLES = "../data/tables"
REPORTS = "../reports/tables"

COLUMNS = [
    "season",
    "game_pk",
    "original_call_code",
    "is_strike_zone",
    "is_call_correct",
    "is_call_correct_confident",
    "distance_from_edge_in",
    "zone_region",
]

g = pd.read_parquet(
    f"{TABLES}/pitches_graded/all_seasons.parquet", columns=COLUMNS
)
g = g[g["is_call_correct"].notna()]

print(g.shape)
print(g.groupby("season").size())

In [ ]:
strict = rate_table(g, ["season"], "is_call_correct")

confident = rate_table(
    g.dropna(subset=["is_call_correct_confident"]),
    ["season"],
    "is_call_correct_confident",
)

by_season = strict.merge(
    confident, on="season", suffixes=("", "_confident")
).sort_values("season")

by_season.round(4)

In [ ]:
in_zone = g["is_strike_zone"].astype(bool)
g["is_called_strike"] = g["original_call_code"] == "C"
g["is_called_ball"] = g["original_call_code"] == "B"

# true ball called a strike
false_strike = rate_table(g[~in_zone], ["season"], "is_called_strike")

# true strike called a ball
false_ball = rate_table(g[in_zone], ["season"], "is_called_ball")

errors = false_strike.merge(
    false_ball, on="season", suffixes=("_false_strike", "_false_ball")
).sort_values("season")

errors.round(4)

In [ ]:
region = rate_table(g, ["season", "zone_region"], "is_call_correct")

region_rate = region.pivot(
    index="zone_region", columns="season", values="rate"
).round(4)

region_n = region.pivot(
    index="zone_region", columns="season", values="trials"
)

print(region_rate)
print()
print(region_n)

In [ ]:
from pathlib import Path

FIGURES = "../reports/figures"
Path(FIGURES).mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    by_season["season"],
    by_season["rate"] * 100,
    marker="o",
    label="All called pitches",
)
ax.plot(
    by_season["season"],
    by_season["rate_confident"] * 100,
    marker="o",
    label="More than 0.5 in from the edge",
)
ax.axvline(
    2025.5,
    color="grey",
    linestyle="--",
    linewidth=1,
    label="2026: ABS zone, midpoint location",
)
ax.set_xlabel("Season")
ax.set_ylabel("Original call correct (%)")
ax.set_title("Plate umpire accuracy by season")
ax.legend()
fig.tight_layout()
fig.savefig(f"{FIGURES}/accuracy_by_season.png", dpi=150)
plt.show()

In [ ]:
by_season.to_csv(f"{REPORTS}/accuracy_by_season.csv", index=False)
errors.to_csv(f"{REPORTS}/error_rates_by_season.csv", index=False)
region.to_csv(f"{REPORTS}/accuracy_by_region_season.csv", index=False)
print("saved 3 tables + 1 figure")

## Takeaways

1. **Umpires got better over the six seasons.** Correct calls: 92.2% (2021) → 92.9% (2025) → 93.5% (2026), with one step back in 2024 (92.9% → 92.6%). Intervals are ±0.1 pp, so the steps are real.
2. **The 2026 step is the biggest, but it uses a different strike zone.** 2026 is graded with the ABS zone and the midpoint location; 2021–2025 with operator-set zones and the front of the plate. Part of the +0.6 pp may be measurement, not umpires.
3. **Misses live on the edge.** Pitches more than half an inch from the edge are called right about 94–96% of the time.
4. **Umpires miss strikes more than balls.** A real strike is called a ball ~10–12% of the time; a real ball is called a strike ~5–7%.
5. **2025 looks different.** Fewest false strikes (4.6%) but most false balls (12.4%) — a tighter called zone. Question for J2 zone maps: umpire behaviour, or how the 2025 zone was recorded?
6. **Hardest spots: high-outside and high-inside** (88–91%); the 2026 gains come from there. Low-middle got worse in 2026 (94.6% → 93.2%) — likely the new bottom-of-zone rule (27% of height), not umpires.